# NB11 Reproducibility Runs — 30 Independent Full-Pipeline Executions

**Project:** Evolving Interpretable Sepsis Mortality Risk Scores via Genetic Programming
**Notebook:** NB11_repro_runs — Reproducibility/Fairness Analysis (supplementary to NB11)


1. **Loss function** — switched to the  pilot-validated custom BCE loss (logit
   space) used in `NB10_gp_symbolic_regression.ipynb`. `gp.predict()` now returns a
   raw logit; `sigmoid()` converts to probability before any metric is computed,
   replacing `np.clip()`.
2. **N_RUNS 10 → 30** —  10 independent runs gives insufficient power for a Bonferroni-corrected one-sample Wilcoxon test; 30 runs
   matches the canonical NB10 marathon's run count.

Results now write to `results/v2_bce/reproducibility_runs/`; the frozen MSE-loss,
10-run baseline is preserved at `results/v1_mse/reproducibility_runs/` and is never
overwritten.

## Purpose

Addresses the concern regarding genuine reproducibility evidence for the model comparison in NB11, rather than bootstrap resampling of a single fixed test set (identified as pseudo-replication — the bootstrap resamples share the same underlying patients and fixed predictions, so they are not independent units of evidence).

This notebook re-runs the **entire pipeline** 30 times, each with a fresh random data partition:
1. A new stratified 70/10/20 train/val/test split — different patients in each run's test set, not a resample of one fixed test set
2. LR, RF, XGB retrained fresh on that run's training data
3. LR_platt, RF_platt fitted fresh (Platt sigmoid on that run's validation set)
4. GP run **once** per split (single PySR call, canonical hyperparameters, custom BCE loss) — not the full 30-run marathon
5. All seven models (including APACHE-IV) evaluated on that run's test set

This produces 30 genuinely independent values per model per metric — a valid basis for a one-sample Wilcoxon test, unlike bootstrap iterations of one fixed split.

**The 30-run GP marathon (NB10) and its canonical formula are completely unaffected by this notebook.** That marathon discovers and selects the formula reported throughout the thesis (NB10–NB14). This notebook is a separate, additive robustness/fairness check — GP here is retrained per split (single run, not the full marathon) specifically because equal treatment with the retrained baselines was required, not because the canonical formula needed re-deriving.

## Design decisions

| Decision | Rationale |
|---|---|
| Single PySR run per split (not the full marathon) | Matches the one-fit-per-split effort given to LR/RF/XGB — fair comparison; a 30-run marathon x 30 splits is infeasible |
| Same canonical PySR hyperparameters (niterations=6000, populations=30, maxsize=24, parsimony=0.001) + custom BCE loss | Consistency with the discovery process used for the canonical formula |
| GP expression selected per run by val-set fitness (AUROC - 0.5xECE), computed on sigmoid-transformed probabilities | Mirrors NB10 Cell 2's per-run selection exactly |
| random_state = run seed (0-29) for split, LR, RF, XGB, and PySR | Each run is a genuinely fresh, independently seeded pipeline execution |
| RF/XGB n_jobs=1 | Avoids Julia threading deadlock on Windows when PySR and sklearn coexist in the same process (established fix from NB13) |
| Resume-safe loop | Each run's outputs are checked before starting; completed runs are skipped automatically if interrupted and re-run |

## Runtime warning

**Each PySR run takes approximately 50–75 minutes** (observed range across the NB10 marathon). For 30 runs this is approximately **25–37 hours total**. This notebook is designed to run unattended over multiple sessions — the loop in Cell 2 is resume-safe and will skip any run whose output already exists on disk.

## Inputs

| File | Description |
|---|---|
| `data/processed/features_curated.parquet` | Full feature matrix (11,164 x 61) — stable across all 30 runs |
| `data/processed/feature_config.json` | MODELLING_COLS (58) and GP_TERMINALS (26) |
| `data/processed/apache4_predictions.csv` | APACHE-IV predicted mortality — full cohort |

## Outputs

| File | Description |
|---|---|
| `results/v2_bce/reproducibility_runs/run_01/` ... `run_30/` | Per-run artefacts: `split_train_val_test.csv`, `model_predictions.csv`, `gp_model.pkl`, `run_metrics.csv` |
| `results/v2_bce/tables/NB11_30run_reproducibility.csv` | Master aggregated table — 30 runs x 7 models, one row each |

## Run-to-seed mapping

`run_NN` (NN = 01..30) uses `random_state = NN - 1` (seeds 0–29) for the split, LR, RF, XGB, and PySR — i.e. `run_01` is seed 0 through `run_30` is seed 29.

In [3]:
%matplotlib inline
from pathlib import Path
import pandas as pd
import numpy as np
import json, pickle, time, sys, warnings
warnings.filterwarnings("ignore")

_nb_dir   = Path().resolve()
PROJECT   = _nb_dir.parent if _nb_dir.name == "notebooks" else _nb_dir
DATA_PROC = PROJECT / "data" / "processed"

# ── Versioned results layout (matches NB10) ───────────────────────────────────
RESULTS_VERSION = "v2_bce"
SHARED    = PROJECT / "results" / "shared"
TABLES    = PROJECT / "results" / RESULTS_VERSION / "tables"
REPRO_DIR = PROJECT / "results" / RESULTS_VERSION / "reproducibility_runs"
REPRO_DIR.mkdir(parents=True, exist_ok=True)

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

# Same custom BCE loss as NB10_gp_symbolic_regression.ipynb Cell 1 — see that
# notebook for the pilot-validation evidence (NB10_pilot_custom_loss.ipynb).
# FOOTNOTE: verbatim identical to the Julia loss specified by the supervisor
# (Teams message, 2026-06-25, "Pilot A / BCE" spec) — char-for-char match confirmed.
BCE_LOSS = ("((pred, tgt) -> let p = 1.0/(1.0+exp(-pred)); "
            "p = clamp(p, 1e-10, 1.0-1e-10); "
            "-(tgt*log(p) + (1.0-tgt)*log(1.0-p)) end)")

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from src.metrics import compute_ece, compute_metrics, calibration_bins

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# ── Primary inputs (stable across all 30 runs — only the split changes) ──────
feat = pd.read_parquet(DATA_PROC / "features_curated.parquet")
with open(DATA_PROC / "feature_config.json") as f:
    cfg = json.load(f)
MODELLING_COLS = cfg["MODELLING_COLS"]
GP_TERMINALS   = cfg["GP_TERMINALS"]
apache_preds   = pd.read_csv(DATA_PROC / "apache4_predictions.csv")

print(f"Results version : {RESULTS_VERSION}  (custom BCE loss, logit-space)")
print(f"Feature frame   : {feat.shape}")
print(f"MODELLING_COLS  : {len(MODELLING_COLS)} features")
print(f"GP_TERMINALS    : {len(GP_TERMINALS)} features")
print(f"REPRO_DIR       : {REPRO_DIR}")

# ── Import PySR (triggers Julia JIT compilation on first run) ──────────────
print()
print("Importing PySR — Julia compilation may take 5-10 min on first run ...")
from pysr import PySRRegressor
print("PySR import complete.")
print("Warming up Julia workers (1-2 min) ...")
_w = PySRRegressor(niterations=1, populations=2, maxsize=7, verbosity=0, progress=False,
                    elementwise_loss=BCE_LOSS)
_w.fit(feat[GP_TERMINALS].iloc[:200, :5],
       feat["hospital_mortality"].to_numpy(dtype=np.float64)[:200])
del _w
print("Julia workers warmed up — Cell 2 is ready.")

Results version : v2_bce  (custom BCE loss, logit-space)
Feature frame   : (11164, 61)
MODELLING_COLS  : 58 features
GP_TERMINALS    : 26 features
REPRO_DIR       : C:\ML PROJECT\sepsis-gp\results\v2_bce\reproducibility_runs

Importing PySR — Julia compilation may take 5-10 min on first run ...
Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
PySR import complete.
Warming up Julia workers (1-2 min) ...
Julia workers warmed up — Cell 2 is ready.


In [2]:
# ═════════════════════════════════════════════════════════════════════════════════════
# 30-run full-pipeline reproducibility loop (resume-safe).
# Each iteration: fresh split -> retrain LR/RF/XGB -> fit Platt -> single PySR run
# (custom BCE loss) -> evaluate all 7 models on that run's test set -> save per-run artefacts.
# ═════════════════════════════════════════════════════════════════════════════════════
N_RUNS = 30

print("30-Run Full-Pipeline Reproducibility (BCE loss)")
print(f"  Each run: fresh 70/10/20 split -> retrain LR/RF/XGB/LR_platt/RF_platt -> single PySR run -> evaluate")
print(f"  PySR config : niterations=6000, populations=30, maxsize=24, BCE loss (canonical hyperparameters)")
print(f"  Resume-safe : completed runs are skipped automatically")
print()

marathon_t0 = time.time()
master_rows = []

for run_idx in range(N_RUNS):
    run_seed = run_idx                      # run_01 -> seed 0, ..., run_30 -> seed 29
    run_name = f"run_{run_idx + 1:02d}"
    run_dir  = REPRO_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = run_dir / "run_metrics.csv"

    if metrics_path.exists():
        print(f"{run_name} (seed={run_seed}): already complete — skipping")
        existing = pd.read_csv(metrics_path)
        existing["run"]  = run_idx + 1
        existing["seed"] = run_seed
        master_rows.append(existing)
        continue

    run_t0 = time.time()
    print(f"{run_name} (seed={run_seed}): starting ...", flush=True)

    # ── Fresh stratified 70/10/20 split ──────────────────────────────
    y_all = feat["hospital_mortality"].to_numpy(dtype=np.float64)
    all_idx = np.arange(len(feat))
    train_full_idx, test_idx = train_test_split(
        all_idx, test_size=0.20, stratify=y_all, random_state=run_seed)
    y_train_full = y_all[train_full_idx]
    train_idx, val_idx = train_test_split(
        train_full_idx, test_size=0.125, stratify=y_train_full, random_state=run_seed)
    # 0.125 of the 80% train_full pool = 10% of the total cohort

    split_arr = np.full(len(feat), "train", dtype=object)
    split_arr[val_idx]  = "val"
    split_arr[test_idx] = "test"
    split_out = feat[["patientunitstayid"]].copy()
    split_out["split"] = split_arr
    split_out.to_csv(run_dir / "split_train_val_test.csv", index=False)

    train_df = feat.iloc[train_idx].reset_index(drop=True)
    val_df   = feat.iloc[val_idx].reset_index(drop=True)
    test_df  = feat.iloc[test_idx].reset_index(drop=True)

    X_train = train_df[MODELLING_COLS].to_numpy(dtype=np.float64)
    y_train = train_df["hospital_mortality"].to_numpy(dtype=np.float64)
    X_val   = val_df[MODELLING_COLS].to_numpy(dtype=np.float64)
    y_val   = val_df["hospital_mortality"].to_numpy(dtype=np.float64)
    X_test  = test_df[MODELLING_COLS].to_numpy(dtype=np.float64)
    y_test  = test_df["hospital_mortality"].to_numpy(dtype=np.float64)

    X_train_gp = train_df[GP_TERMINALS].copy()
    X_val_gp   = val_df[GP_TERMINALS].copy()
    X_test_gp  = test_df[GP_TERMINALS].copy()

    print(f"  Split: train={len(train_df):,} ({y_train.mean()*100:.2f}%)  "
          f"val={len(val_df):,} ({y_val.mean()*100:.2f}%)  "
          f"test={len(test_df):,} ({y_test.mean()*100:.2f}%)")

    # ── Train LR, RF, XGB (canonical hyperparameters, run-specific seed) ─────
    spw = (y_train == 0).sum() / (y_train == 1).sum()
    models = {
        "LR": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(C=1.0, class_weight="balanced",
                                        solver="lbfgs", max_iter=1000,
                                        random_state=run_seed))
        ]),
        "RF": RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                      class_weight="balanced", random_state=run_seed,
                                      n_jobs=1),
        "XGB": XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                              subsample=0.8, colsample_bytree=0.8,
                              scale_pos_weight=spw, random_state=run_seed,
                              verbosity=0, eval_metric="logloss", n_jobs=1),
    }
    fitted, test_preds = {}, {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        fitted[name] = model
        test_preds[name] = model.predict_proba(X_test)[:, 1]

    # ── Platt scaling for LR and RF (fit on val, mirrors NB08) ───────────
    for base_name in ["LR", "RF"]:
        p_val_raw  = np.clip(fitted[base_name].predict_proba(X_val)[:, 1], 1e-7, 1 - 1e-7)
        logit_val  = np.log(p_val_raw / (1 - p_val_raw)).reshape(-1, 1)
        platt = LogisticRegression(solver="lbfgs", max_iter=1000)
        platt.fit(logit_val, y_val)
        p_test_raw = np.clip(test_preds[base_name], 1e-7, 1 - 1e-7)
        logit_test = np.log(p_test_raw / (1 - p_test_raw)).reshape(-1, 1)
        test_preds[f"{base_name}_platt"] = platt.predict_proba(logit_test)[:, 1]

    # ── Single PySR run (canonical hyperparameters + BCE loss, run-specific seed) ──
    gp = PySRRegressor(
        niterations      = 6000,
        populations      = 30,
        maxsize          = 24,
        parsimony        = 0.001,
        binary_operators = ["+", "-", "*", "/", "max", "min"],
        unary_operators  = ["log", "sqrt", "exp", "abs"],
        elementwise_loss = BCE_LOSS,
        model_selection  = "best",
        random_state     = run_seed,
        verbosity        = 0,
        progress         = False,
    )
    gp.fit(X_train_gp, y_train)

    # gp.predict() returns the raw logit under BCE loss — sigmoid-transform before
    # computing probability-scale metrics, same as NB10 Cell 2.
    pareto_rows = []
    for idx, eq_row in gp.equations_.iterrows():
        try:
            raw_val  = gp.predict(X_val_gp, index=idx)
            raw_val  = np.where(np.isfinite(raw_val), raw_val, 0.0)
            prob_val = sigmoid(raw_val)
            m = compute_metrics(y_val, prob_val, label=f"GP_c{int(eq_row['complexity'])}",
                                 fitness_alpha=1.0, fitness_beta=0.5)
            m["complexity"] = int(eq_row["complexity"])
            m["equation"]   = str(eq_row["equation"])
            pareto_rows.append(m)
        except Exception as e:
            print(f"    c={eq_row['complexity']} skipped: {e}")
    pareto_df = pd.DataFrame(pareto_rows)
    best_idx  = pareto_df["fitness"].idxmax()
    best      = pareto_df.loc[best_idx]
    eq_idx    = gp.equations_[gp.equations_["complexity"] == int(best["complexity"])].index[0]

    gp_raw_test = gp.predict(X_test_gp, index=eq_idx)
    gp_raw_test = np.where(np.isfinite(gp_raw_test), gp_raw_test, 0.0)
    test_preds["GP"] = sigmoid(gp_raw_test)

    with open(run_dir / "gp_model.pkl", "wb") as f:
        pickle.dump({"model": gp, "complexity": int(best["complexity"]),
                     "equation": str(best["equation"]),
                     "val_fitness": float(best["fitness"])}, f)

    # ── APACHE-IV on this run's test subset ───────────────────────
    test_ids = set(test_df["patientunitstayid"])
    apache_test = apache_preds[
        apache_preds["patientunitstayid"].isin(test_ids) & apache_preds["apache4_pred"].notna()
    ]

    # ── Evaluate all 7 models on the test set ─────────────────────
    run_metric_rows = []
    for name in ["GP", "LR", "RF", "XGB", "LR_platt", "RF_platt"]:
        m = compute_metrics(y_test, test_preds[name], label=name)
        m["model"] = name
        if name == "GP":
            m["complexity"] = int(best["complexity"])
            m["equation"]   = str(best["equation"])
        run_metric_rows.append(m)
    m_ap = compute_metrics(
        apache_test["hospital_mortality"].to_numpy(),
        apache_test["apache4_pred"].to_numpy(), label="APACHE-IV")
    m_ap["model"] = "APACHE-IV"
    run_metric_rows.append(m_ap)
    run_metrics_df = pd.DataFrame(run_metric_rows)
    run_metrics_df.to_csv(metrics_path, index=False)

    # ── Save test-set predictions for all models ──────────────────
    preds_out = test_df[["patientunitstayid", "hospitalid", "hospital_mortality"]].copy()
    for name in ["GP", "LR", "RF", "XGB", "LR_platt", "RF_platt"]:
        preds_out[name.lower() + "_pred"] = test_preds[name]
    preds_out.to_csv(run_dir / "model_predictions.csv", index=False)

    elapsed = (time.time() - run_t0) / 60
    print(f"  GP: c={int(best['complexity'])}  val_fitness={best['fitness']:.4f}")
    print(f"    equation = {best['equation']}")
    print(f"  Test metrics (AUROC/ECE): " +
          "  ".join(f"{r['model']}={r['auroc']:.3f}/{r['ece_10bin']:.3f}"
                     for _, r in run_metrics_df.iterrows()))
    print(f"  {run_name} done in {elapsed:.1f} min")
    print()

    run_metrics_df["run"]  = run_idx + 1
    run_metrics_df["seed"] = run_seed
    master_rows.append(run_metrics_df)

total_elapsed = (time.time() - marathon_t0) / 3600
print("=" * 60)
print(f"30-run reproducibility loop complete: {total_elapsed:.2f} hours")
print("=" * 60)

30-Run Full-Pipeline Reproducibility (BCE loss)
  Each run: fresh 70/10/20 split -> retrain LR/RF/XGB/LR_platt/RF_platt -> single PySR run -> evaluate
  PySR config : niterations=6000, populations=30, maxsize=24, BCE loss (canonical hyperparameters)
  Resume-safe : completed runs are skipped automatically

run_01 (seed=0): starting ...
  Split: train=7,814 (16.88%)  val=1,117 (16.92%)  test=2,233 (16.88%)
  GP: c=24  val_fitness=0.7539
    equation = log(sqrt(bun) / ((map_mean / (lactate_max + bilirubin)) * (map_mean / age_numeric))) - min(log(pf_ratio * (gcs_total * (0.07448286 / heartrate))), pf_ratio_miss)
  Test metrics (AUROC/ECE): GP=0.736/0.024  LR=0.762/0.245  RF=0.771/0.110  XGB=0.757/0.106  LR_platt=0.762/0.018  RF_platt=0.771/0.016  APACHE-IV=0.695/0.064
  run_01 done in 129.6 min

run_02 (seed=1): starting ...
  Split: train=7,814 (16.88%)  val=1,117 (16.92%)  test=2,233 (16.88%)
  GP: c=15  val_fitness=0.7098
    equation = (((age_numeric / map_mean) * sqrt(lactate_max)) +

### Findings — Cell 2: 30-Run Full-Pipeline Reproducibility Loop

The 30-run full-pipeline reproducibility loop completed in approximately **66 hours** (mean ≈ 132 min per run, reflecting the combined cost of retraining LR/RF/XGB, fitting Platt scalers, and running a full PySR search at canonical budget — niterations=6,000, populations=30 — for each independent pipeline execution). All 30 runs completed successfully; per-run artefacts (split CSV, model predictions, GP model pickle, run metrics CSV) were saved to `results/v2_bce/reproducibility_runs/run_01/` through `run_30/`.

**GP per-run test-set metrics (each row is a genuinely independent pipeline execution on a fresh data split):**

| Run | AUROC | ECE | Brier | Complexity |
|---|---|---|---|---|
| run_01 | 0.7355 | 0.0244 | 0.1229 | 24 |
| run_02 | 0.7123 | 0.0202 | 0.1245 | 15 |
| run_03 | 0.7175 | 0.0162 | 0.1239 | 20 |
| run_04 | 0.7301 | 0.0100 | 0.1233 | 21 |
| run_05 | 0.7298 | 0.0124 | 0.1218 | 23 |
| run_06 | 0.7477 | 0.0168 | 0.1196 | 24 |
| run_07 | 0.7710 | 0.0174 | 0.1170 | 22 |
| run_08 | 0.6616 | 0.0189 | 0.1299 | 6 |
| run_09 | 0.7207 | 0.0194 | 0.1261 | 22 |
| run_10 | 0.7177 | 0.0193 | 0.1248 | 24 |
| run_11 | 0.7127 | 0.0190 | 0.1270 | 19 |
| run_12 | 0.7604 | 0.0220 | 0.1196 | 20 |
| run_13 | 0.7193 | 0.0229 | 0.1264 | 19 |
| run_14 | 0.7396 | 0.0203 | 0.1221 | 18 |
| run_15 | 0.7486 | 0.0183 | 0.1192 | 24 |
| run_16 | 0.7408 | 0.0164 | 0.1230 | 24 |
| run_17 | 0.7537 | 0.0161 | 0.1187 | 23 |
| run_18 | 0.7375 | 0.0147 | 0.1207 | 24 |
| run_19 | 0.7059 | 0.0118 | 0.1251 | 18 |
| run_20 | 0.7183 | 0.0102 | 0.1243 | 23 |
| run_21 | 0.7532 | 0.0147 | 0.1198 | 22 |
| run_22 | 0.7384 | 0.0083 | 0.1234 | 22 |
| run_23 | 0.7266 | 0.0193 | 0.1231 | 17 |
| run_24 | 0.7282 | 0.0172 | 0.1250 | 24 |
| run_25 | 0.7618 | 0.0157 | 0.1174 | 17 |
| run_26 | 0.7428 | 0.0136 | 0.1218 | 24 |
| run_27 | 0.7224 | 0.0122 | 0.1253 | 15 |
| run_28 | 0.7395 | 0.0096 | 0.1208 | 23 |
| run_29 | 0.7678 | 0.0148 | 0.1185 | 23 |
| run_30 | 0.7553 | 0.0125 | 0.1200 | 18 |

**GP distribution summary across 30 runs:**

| Metric | Mean | SD | Min (run) | Max (run) |
|---|---|---|---|---|
| AUROC | 0.7339 | 0.0221 | 0.6616 (run_08) | 0.7710 (run_07) |
| ECE | 0.0162 | 0.0041 | 0.0083 (run_22) | 0.0244 (run_01) |
| Brier | 0.1225 | 0.0031 | 0.1170 (run_07) | 0.1299 (run_08) |

Selected expression complexity ranged from 6 (run_08) to 24 (8 of 30 runs). Excluding run_08, complexity ranged from 15 to 24 across the remaining 29 runs, consistent with the complexity distribution observed in the canonical 30-run GP marathon.

**Critical observation — run_08 search failure.** Run_08 produced a complexity-6 expression (two features: `sqrt(lactate_max)` and `pf_ratio_miss`), with an AUROC of 0.6616 — the single severe outlier across the full 30-run distribution, and the only run falling below APACHE-IV's mean AUROC. Inspection of run_08's other models confirmed this was not a hard data split: LR (0.7620), RF (0.7752), XGB (0.7604), and APACHE-IV (0.7116) all performed at or above their respective across-run averages. The failure was therefore specific to the GP search on that seed/split combination — the evolutionary search converged prematurely to an overly simple expression without developing a richer structure. This occurred once in 30 runs (3.3%), representing an inherent stochastic risk of evolutionary search methods rather than a systematic failure of the procedure. Run_19 (AUROC 0.7059, complexity 18) was the next-weakest result but retained a reasonable expression structure, suggesting it reflects a harder-than-average test partition rather than a degenerate search.

The full 7-model comparison across all 30 runs is aggregated and statistically tested in Cell 3.


In [2]:
# ═════════════════════════════════════════════════════════════════════════════════════
# Aggregate all 30 runs into the master reproducibility table.
# Self-contained: reads from saved run_metrics.csv files on disk rather than
# relying on master_rows in memory — safe to run after kernel restart.
# ═════════════════════════════════════════════════════════════════════════════════════
import pandas as pd
from pathlib import Path

_nb_dir  = Path().resolve()
PROJECT  = _nb_dir.parent if _nb_dir.name == "notebooks" else _nb_dir
RESULTS_VERSION = "v2_bce"
TABLES    = PROJECT / "results" / RESULTS_VERSION / "tables"
REPRO_DIR = PROJECT / "results" / RESULTS_VERSION / "reproducibility_runs"
TABLES.mkdir(parents=True, exist_ok=True)

rows = []
for run_dir in sorted(REPRO_DIR.glob("run_*")):
    mpath = run_dir / "run_metrics.csv"
    if not mpath.exists():
        print(f"WARNING: {run_dir.name} has no run_metrics.csv — skipping")
        continue
    df = pd.read_csv(mpath)
    df["run"]  = int(run_dir.name.split("_")[1])
    df["seed"] = int(run_dir.name.split("_")[1]) - 1
    rows.append(df)

master_df = pd.concat(rows, ignore_index=True)
out_master = TABLES / "NB11_30run_reproducibility.csv"
master_df.to_csv(out_master, index=False)
print(f"Saved: {out_master}")
print(f"  {len(master_df)} rows = {master_df['run'].nunique()} runs x {master_df['model'].nunique()} models")
print()

print("Mean +/- SD across 30 runs, per model:")
summary = (master_df.groupby("model")[
    ["auroc", "auprc", "brier", "ece_10bin", "cal_slope", "cal_intercept"]
].agg(["mean", "std"]))
print(summary.round(4).to_string())

Saved: C:\ML PROJECT\sepsis-gp\results\v2_bce\tables\NB11_30run_reproducibility.csv
  210 rows = 30 runs x 7 models

Mean +/- SD across 30 runs, per model:
            auroc           auprc           brier         ece_10bin         cal_slope         cal_intercept        
             mean     std    mean     std    mean     std      mean     std      mean     std          mean     std
model                                                                                                              
APACHE-IV  0.7029  0.0155  0.3778  0.0209  0.1334  0.0034    0.0603  0.0069    0.4119  0.0551       -0.9463  0.0730
GP         0.7339  0.0221  0.4114  0.0289  0.1225  0.0031    0.0162  0.0041    0.9638  0.1031       -0.0540  0.1346
LR         0.7715  0.0141  0.4608  0.0226  0.1893  0.0033    0.2508  0.0040    0.9136  0.0713       -1.5804  0.0266
LR_platt   0.7715  0.0141  0.4608  0.0226  0.1169  0.0026    0.0166  0.0053    1.0127  0.1163        0.0080  0.1585
RF         0.7772  0.0138  0.463

### Findings — Cell 3: 30-Run Distribution Summary

The master reproducibility table was saved to `results/v2_bce/tables/NB11_30run_reproducibility.csv` (210 rows = 30 runs × 7 models).

**Mean ± SD across 30 runs, per model:**

| Model | AUROC mean ± SD | ECE mean ± SD | Brier mean ± SD | Cal slope mean ± SD |
|---|---|---|---|---|
| GP | 0.7339 ± 0.0221 | **0.0162 ± 0.0041** | 0.1225 ± 0.0031 | 0.964 ± 0.103 |
| LR | 0.7715 ± 0.0141 | 0.2508 ± 0.0040 | 0.1893 ± 0.0033 | 0.914 ± 0.071 |
| RF | 0.7772 ± 0.0138 | 0.1132 ± 0.0039 | 0.1298 ± 0.0022 | 1.393 ± 0.111 |
| XGB | 0.7655 ± 0.0123 | 0.1110 ± 0.0048 | 0.1395 ± 0.0032 | 0.751 ± 0.053 |
| LR_platt | 0.7715 ± 0.0141 | **0.0166 ± 0.0053** | 0.1169 ± 0.0026 | 1.013 ± 0.116 |
| RF_platt | 0.7772 ± 0.0138 | 0.0179 ± 0.0074 | 0.1166 ± 0.0026 | 1.006 ± 0.151 |
| APACHE-IV | 0.7029 ± 0.0155 | 0.0603 ± 0.0069 | 0.1334 ± 0.0034 | 0.412 ± 0.055 |

**Calibration — the central finding, confirmed at 30-run scale.** GP ECE (0.0162) is nearly indistinguishable from LR_platt (0.0166) and RF_platt (0.0179) across 30 genuinely independent pipeline executions. All three calibration slopes cluster near unity (GP 0.964, LR_platt 1.013, RF_platt 1.006), and all three calibration intercepts are near zero, confirming well-centred probability estimates as a stable, reproducible property of these three models. GP achieves this without any post-hoc recalibration step. The raw uncalibrated baselines show the expected calibration degradation from class reweighting: RF cal slope 1.393 (overconfident at high predicted risk), XGB slope 0.751 with large negative intercept (systematic over-prediction), raw LR ECE 0.251 (probability inflation from class reweighting). APACHE-IV's calibration slope (0.412) reflects well-documented over-prediction of risk relative to this cohort's 16.9% observed mortality rate.

**Discrimination — GP deficit is real and reproducible across all 30 runs.** GP AUROC (0.7339) consistently trails RF/RF_platt (0.7772, gap ≈ 0.043), LR/LR_platt (0.7715, gap ≈ 0.038), and XGB (0.7655, gap ≈ 0.032) across the full distribution. GP consistently outperforms APACHE-IV (0.7029, advantage ≈ 0.031). GP's AUROC standard deviation (0.0221) is wider than every other model's (0.012–0.022), driven primarily by two underperforming runs identified in Cell 2 — run_08 (AUROC 0.6616, degenerate complexity-6 search failure) and run_19 (AUROC 0.7059). Formal Wilcoxon signed-rank tests on these distributions are presented in the sensitivity-check analysis.


In [4]:
N_RUNS = 30


In [5]:
# ═════════════════════════════════════════════════════════════════════════════════════
# Sensitivity check on the GP selection criterion (no new PySR training — pure inference).
#
# Reloads each of the 30 already-pickled GP models and re-evaluates their FULL Pareto
# front on that run's val set under three selection criteria: the canonical
# AUROC - 0.5*ECE fitness, AUROC alone, and Brier alone. For each criterion, the
# selected expression is evaluated once on the test set (test is never used for
# selection — only for this final reporting step, consistent with the methods sign-off).
# gp.predict() returns the raw logit under BCE loss — sigmoid-transform before computing
# any probability-scale metric, same as Cell 2.
#
# Purpose: confirm the 0.5 weighting on ECE is not driving the headline conclusion.
# If all three criteria select similarly-performing expressions, the 0.5 weight is
# not load-bearing. If they diverge substantially, the weighting choice needs
# explicit justification or a different criterion.
#
# Also checks: does a run's validation performance predict its test performance?
# This distinguishes "search failure" (bad val AND bad test -> the search genuinely
# struggled on that split) from "val-test instability" (good val but bad test ->
# the selection criterion picked something that didn't generalise even within
# that split). This determines whether best-of-N (re-running the search more
# times) is the right fix, or whether a different problem is present.
# ═════════════════════════════════════════════════════════════════════════════════════
sensitivity_rows = []

for run_idx in range(N_RUNS):
    run_seed = run_idx
    run_name = f"run_{run_idx + 1:02d}"
    run_dir  = REPRO_DIR / run_name

    # ── Reconstruct this run's exact val/test split (same seed, same logic) ──────
    y_all = feat["hospital_mortality"].to_numpy(dtype=np.float64)
    all_idx = np.arange(len(feat))
    train_full_idx, test_idx = train_test_split(
        all_idx, test_size=0.20, stratify=y_all, random_state=run_seed)
    y_train_full = y_all[train_full_idx]
    train_idx, val_idx = train_test_split(
        train_full_idx, test_size=0.125, stratify=y_train_full, random_state=run_seed)

    val_df  = feat.iloc[val_idx].reset_index(drop=True)
    test_df = feat.iloc[test_idx].reset_index(drop=True)
    X_val_gp  = val_df[GP_TERMINALS].copy()
    y_val     = val_df["hospital_mortality"].to_numpy(dtype=np.float64)
    X_test_gp = test_df[GP_TERMINALS].copy()
    y_test    = test_df["hospital_mortality"].to_numpy(dtype=np.float64)

    # ── Reload the already-fitted GP model (no retraining) ──────────────────────
    with open(run_dir / "gp_model.pkl", "rb") as f:
        saved = pickle.load(f)
    gp = saved["model"]

    # ── Re-evaluate the full Pareto front on val ─────────────────────────────────
    pareto_rows = []
    for idx, eq_row in gp.equations_.iterrows():
        try:
            raw_val  = gp.predict(X_val_gp, index=idx)
            raw_val  = np.where(np.isfinite(raw_val), raw_val, 0.0)
            prob_val = sigmoid(raw_val)
            m = compute_metrics(y_val, prob_val, label=f"c{int(eq_row['complexity'])}",
                                 fitness_alpha=1.0, fitness_beta=0.5)
            m["complexity"] = int(eq_row["complexity"])
            m["eq_idx"] = idx
            m["equation"] = str(eq_row["equation"])
            pareto_rows.append(m)
        except Exception:
            continue
    pdf = pd.DataFrame(pareto_rows)

    # ── Select under three criteria, evaluate selected expression on test ───────
    criteria = {
        "auroc_minus_half_ece": ("fitness", "max"),
        "auroc_only":            ("auroc",   "max"),
        "brier_only":            ("brier",   "min"),
    }
    for crit_name, (col, direction) in criteria.items():
        best_idx = pdf[col].idxmax() if direction == "max" else pdf[col].idxmin()
        best_row = pdf.loc[best_idx]
        eq_idx   = int(best_row["eq_idx"])

        gp_raw_test = gp.predict(X_test_gp, index=eq_idx)
        gp_raw_test = np.where(np.isfinite(gp_raw_test), gp_raw_test, 0.0)
        gp_p_test   = sigmoid(gp_raw_test)
        test_m = compute_metrics(y_test, gp_p_test, label=crit_name)

        sensitivity_rows.append(dict(
            run=run_idx + 1, criterion=crit_name, complexity=int(best_row["complexity"]),
            equation=best_row["equation"],
            val_auroc=best_row["auroc"], val_ece=best_row["ece_10bin"], val_brier=best_row["brier"],
            test_auroc=test_m["auroc"], test_ece=test_m["ece_10bin"], test_brier=test_m["brier"],
        ))

sens_df = pd.DataFrame(sensitivity_rows)
out_sens = TABLES / "NB11_30run_gp_criterion_sensitivity.csv"
sens_df.to_csv(out_sens, index=False)

print("GP selection-criterion sensitivity check (pure inference, no new PySR training)")
print(f"Saved: {out_sens}  ({len(sens_df)} rows = {N_RUNS} runs x 3 criteria)")
print()

pivot_auroc = sens_df.pivot(index="run", columns="criterion", values="test_auroc")
pivot_ece   = sens_df.pivot(index="run", columns="criterion", values="test_ece")
pivot_cplx  = sens_df.pivot(index="run", columns="criterion", values="complexity")

print("Test AUROC by selection criterion (rows = runs):")
print(pivot_auroc.round(4).to_string())
print()
print("Test ECE by selection criterion (rows = runs):")
print(pivot_ece.round(4).to_string())
print()
print("Selected complexity by criterion (rows = runs):")
print(pivot_cplx.to_string())
print()

print("Mean +/- SD across 30 runs, by criterion:")
summary = sens_df.groupby("criterion")[["test_auroc", "test_ece", "test_brier"]].agg(["mean", "std"])
print(summary.round(4).to_string())

# ═════════════════════════════════════════════════════════════════════════════════════
# Diagnostic: does low validation performance predict low test performance?
# Distinguishes "search failure" (bad val AND bad test -> genuine search difficulty
# on that split) from "val-test instability" (good val but bad test -> selection
# picked something that did not generalise even within that split's own data).
# ═════════════════════════════════════════════════════════════════════════════════════
canon = sens_df[sens_df["criterion"] == "auroc_minus_half_ece"].sort_values("run")
canon = canon.set_index("run")[
    ["complexity", "val_auroc", "val_ece", "test_auroc", "test_ece", "test_brier", "equation"]
].copy()
canon["val_minus_test_auroc"] = canon["val_auroc"] - canon["test_auroc"]

print()
print("=" * 70)
print("Val-vs-test correspondence diagnostic (canonical auroc_minus_half_ece criterion)")
print("=" * 70)
print(canon[["complexity", "val_auroc", "val_ece", "test_auroc", "test_ece",
             "val_minus_test_auroc"]].round(4).to_string())
print()
print("Selected equations per run:")
for run_n, row in canon.iterrows():
    print(f"  Run {run_n} (c={row['complexity']}): {row['equation']}")
print()

val_test_corr = canon["val_auroc"].corr(canon["test_auroc"])
print(f"Correlation between val AUROC and test AUROC across the {N_RUNS} runs: r = {val_test_corr:.3f}")
print()
print("Interpretation guide:")
print("  - Strong positive correlation + small val-minus-test gaps")
print("      -> validation predicts test well; weak runs reflect genuine search")
print("         difficulty on that split (favours best-of-N as the right fix)")
print("  - Weak/negative correlation or large, inconsistent gaps")
print("      -> validation does not reliably predict test; weak runs may reflect")
print("         val-test instability rather than search failure (best-of-N alone")
print("         would not fully resolve this)")

GP selection-criterion sensitivity check (pure inference, no new PySR training)
Saved: C:\ML PROJECT\sepsis-gp\results\v2_bce\tables\NB11_30run_gp_criterion_sensitivity.csv  (90 rows = 30 runs x 3 criteria)

Test AUROC by selection criterion (rows = runs):
criterion  auroc_minus_half_ece  auroc_only  brier_only
run                                                    
1                        0.7355      0.7355      0.7381
2                        0.7123      0.7333      0.7333
3                        0.7175      0.7175      0.7265
4                        0.7301      0.7301      0.7301
5                        0.7298      0.7298      0.7259
6                        0.7477      0.7477      0.7477
7                        0.7710      0.7710      0.7710
8                        0.6616      0.6616      0.6880
9                        0.7207      0.7207      0.7207
10                       0.7177      0.7177      0.7177
11                       0.7127      0.7049      0.7127
12             

### Findings — Sensitivity Check and Val-vs-Test Correspondence Diagnostic

---

**GP selection-criterion sensitivity check (30 runs × 3 criteria, pure inference — no new PySR training)**

Results saved to `results/v2_bce/tables/NB11_30run_gp_criterion_sensitivity.csv` (90 rows = 30 runs × 3 criteria).

| Selection criterion | Test AUROC mean ± SD | Test ECE mean ± SD | Test Brier mean ± SD |
|---|---|---|---|
| AUROC − 0.5×ECE (canonical) | 0.7339 ± 0.0221 | 0.0162 ± 0.0041 | 0.1225 ± 0.0031 |
| AUROC only | 0.7337 ± 0.0211 | 0.0165 ± 0.0049 | 0.1226 ± 0.0030 |
| Brier only | 0.7363 ± 0.0177 | 0.0172 ± 0.0054 | 0.1223 ± 0.0027 |

**The 0.5 weighting on ECE is not driving the procedure's discrimination or calibration pattern at the 30-run scale.** The canonical criterion and AUROC-only criterion produce nearly identical mean test performance (AUROC 0.7339 vs 0.7337, ECE 0.0162 vs 0.0165), confirming that the ECE component of the composite fitness function neither helps nor harms expression selection in any practically meaningful way. Selecting by Brier alone produces a marginally higher mean test AUROC (0.7363, +0.002) and tighter spread (std 0.0177 vs 0.0221) but slightly weaker ECE (0.0172 vs 0.0162), representing a small, consistent shift in the discrimination-calibration trade-off rather than a clear improvement. Crucially, selecting explicitly on Brier does not produce meaningfully better test-set Brier (0.1223 vs 0.1225) — consistent with validation-set selection not reliably transferring to test regardless of which metric is used.

One noteworthy observation specific to run_08 (the degenerate complexity-6 run): the Brier-only criterion selected a complexity-9 expression instead of complexity-6, yielding test AUROC 0.6880 versus 0.6616 under the canonical criterion — an improvement of 0.026 AUROC points. This indicates that more competitive expressions were available on run_08's Pareto front and were correctly found by an alternative criterion; the canonical criterion's selection of the simpler expression was a val-set artefact rather than a fundamental search failure. This partly qualifies the severity of run_08 as a "degenerate search" — the search did find reasonable higher-complexity candidates, but the validation set (n≈1,117) was not large enough to rank them reliably above the simpler one.

---

**Val-vs-test correspondence diagnostic**

Validation AUROC and test AUROC were compared directly for the canonical criterion across all 30 runs. The correlation between them was **r = −0.126** — essentially zero and weakly negative, confirming that the validation set does not reliably predict test-set AUROC across this run distribution.

Val-minus-test AUROC gaps ranged from −0.066 (run_29: val 0.7018, test 0.7678 — the run with the lowest validation performance produced the second-best test performance) to +0.096 (run_20: val 0.8139, test 0.7183 — the run with the highest validation AUROC produced only middling test performance). The magnitude and direction of these gaps are inconsistent and unpredictable, confirming the noise-dominated character of val-set selection at this scale: the validation set (n≈1,117, ≈190 mortality events) is simply too small to reliably distinguish among Pareto-front candidates whose AUROC differences are of the same order of magnitude as the sampling noise in an AUROC estimate from that sample.

This is the same conclusion reached in the 10-run analysis (r ≈ −0.07), now confirmed with greater statistical stability at 30 runs and a slightly more pronounced negative tendency. As discussed in that earlier analysis, this finding suggests that best-of-N selection using the same validation-set criterion would not systematically improve test-set performance — selecting the winner of a larger candidate pool by a noisy proxy increases the risk of selecting a formula that was merely lucky on validation rather than genuinely more transferable.

---

**Feature consistency across 30 independently evolved expressions**

Despite 30 structurally distinct formulas (complexity 6–24, varying mathematical operations), the same small set of variables consistently dominated the search across runs: `lactate_max` (29/30 runs), `map_mean` (28/30), `age_numeric` (26/30), `bun` (25/30), `vent` (23/30), `bilirubin` and `pf_ratio` (~22/30 each), and `platelets_min` (19/30). The MNAR missingness indicator `pf_ratio_miss` was selected in 10 of 30 runs (33%), confirming that the GP search treats it as a genuinely informative predictor rather than noise in a meaningful fraction of executions — even though the NB10 canonical formula did not include it.

This convergence on the same clinically coherent variable set, across 30 independently seeded searches on 30 independently drawn data splits, is the strongest evidence available in this research that the identified predictors reflect a genuine, stable signal in the data rather than an artefact of any one search or partition. The clinical interpretation is consistent throughout: organ dysfunction (bilirubin, BUN), haemodynamic status (MAP, ventilation), metabolic failure (lactate), demographic risk modifier (age), and gas-exchange reserve (PF ratio) appear together in nearly every formula the evolutionary search independently discovers.
